# Notebook 32 -- Luyben Summary Statistics

Design and validate the 65-D feature vector:
1. Compute summaries for all 360 windows from nb31.
2. PCA and t-SNE visualisation -- check scenario separability.
3. Mutual information ranking.
4. Physics-informed feature sanity checks.


In [ ]:
import sys; sys.path.insert(0, '../src')
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from cstr_sbi.luyben.summaries import compute_summary_statistics_batch, FEATURE_NAMES, N_FEATURES

d = np.load('../data/luyben_observations.npz')
obs = d['x']        # (360, 120, 8)
theta = d['theta']  # (360, 8)
sc_ids = d['scenario_id']
t = d['t']          # (120,)

print(f'Loaded {obs.shape[0]} windows, {obs.shape[1]} timesteps, {obs.shape[2]} channels')
print(f'Computing {N_FEATURES}-D summary statistics ...')

import jax.numpy as jnp
summaries = np.asarray(compute_summary_statistics_batch(jnp.array(obs), jnp.array(t)))
print(f'Summaries shape: {summaries.shape}')
print(f'NaN count: {np.isnan(summaries).sum()}')


In [ ]:
# PCA
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

scaler = StandardScaler()
S_scaled = scaler.fit_transform(summaries)
S_scaled = np.nan_to_num(S_scaled, nan=0.0)

pca = PCA(n_components=4)
S_pca = pca.fit_transform(S_scaled)
print(f'PCA explained variance: {pca.explained_variance_ratio_.cumsum()}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
sc_unique = np.unique(sc_ids)
colors = plt.cm.tab20(np.linspace(0, 1, len(sc_unique)))
for sc_id, color in zip(sc_unique, colors):
    mask = sc_ids == sc_id
    axes[0].scatter(S_pca[mask, 0], S_pca[mask, 1], c=[color], label=f'L{sc_id}', alpha=0.7, s=20)
    axes[1].scatter(S_pca[mask, 2], S_pca[mask, 3], c=[color], label=f'L{sc_id}', alpha=0.7, s=20)
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2'); axes[0].legend(fontsize=6, ncol=3)
axes[1].set_xlabel('PC3'); axes[1].set_ylabel('PC4')
fig.suptitle('PCA of 65-D summary statistics -- Luyben scenarios')
plt.show()


In [ ]:
# Mutual information with alpha (most interesting parameter)
from sklearn.feature_selection import mutual_info_regression

alpha_true = theta[:, 0]
mi_scores = mutual_info_regression(S_scaled, alpha_true, random_state=0)

top_idx = np.argsort(mi_scores)[::-1][:15]
print('Top 15 features by MI with alpha:')
for i in top_idx:
    print(f'  {FEATURE_NAMES[i]:30s}  MI = {mi_scores[i]:.4f}')
